# Add `is_misdemeanor` and `has_warrant` Columns

Adds two derived columns to checkpoint14 and saves checkpoint15.

- **`is_misdemeanor`**: `True` if every charge in the row classifies as `Misdemeanor` under MA law; `False` if any charge is `Felony`, `Either`, or `Unknown`; `NaN` for rows with no charges.
- **`has_warrant`**: `True` if any individual charge in the original `Charges` field carried a warrant prefix (bench warrant, default warrant, standard warrant, capias, child in need); `False` otherwise; `NaN` for rows with no charges.

**Input:** `data/checkpoints/checkpoint14_standardized_charges.csv`  
**Output:** `data/checkpoints/checkpoint15_misdemeanor_warrant.csv`

### Imports & Paths

In [9]:
import os
import numpy as np
import pandas as pd

from standardize_charges import extract_warrant_type

NOTEBOOK_DIR = os.getcwd()

DATA_DIR = os.path.join(
    NOTEBOOK_DIR,
    "..",
    "data",
    "missing_dates_csv",
)

CHECKPOINTS_DIR = os.path.join(DATA_DIR, "md_checkpoints")

IN_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint14_standardized_charges.csv",
)

OUT_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint15_misdemeanor_warrant.csv",
)


REPO_ROOT = os.path.abspath(
    os.path.join(NOTEBOOK_DIR, "..", "..")
)

LOOKUP_PATH = os.path.join(
    REPO_ROOT,
    "streamlit-app",
    "unique_charges_standardized.csv",
)

### Load checkpoint14

In [6]:
df = pd.read_csv(IN_PATH, low_memory=False)
display(df.head())
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Rows with charges: {df['cleaned_charges'].notna().sum()}")
df.head()

,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,latitude,longitude,...,person_id,category,Year,crime_severity,_DOB_raw,_Date_raw,DOB_dt,Date_dt,statutes,cleaned_charges
0,2018-01-01 00:01:14,NOISE ORD,3 HARRIMAN ST,Yes,NaN,07/03/1974,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,3 HARRIMAN ST,42.718522,-71.148148,...,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,Public Disturbances,2018,Non-Serious,07/03/1974,2018-01-01 00:01:14,1974-07-03,2018-01-01 00:01:14,NaN,a&b on family / household member / intimate pa...
1,2018-01-01 00:08:38,LOUD NOISE,1 HARRIMAN ST,Yes,NaN,01/21/1979,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,1 HARRIMAN ST FL 2,42.718522,-71.148148,...,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,Public Disturbances,2018,Non-Serious,01/21/1979,2018-01-01 00:08:38,1979-01-21,2018-01-01 00:08:38,NaN,a&b on family / household member / intimate pa...
2,2018-01-01 00:11:17,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,16 ALLEN ST,42.710782,-71.151911,...,NaN,Fire and Arson Incidents,2018,Non-Serious,NaN,2018-01-01 00:11:17,NaN,2018-01-01 00:11:17,NaN,NaN
3,2018-01-01 00:14:53,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,11 SUMMER ST,42.711117,-71.153015,...,NaN,Public Disturbances,2018,Non-Serious,NaN,2018-01-01 00:14:53,NaN,2018-01-01 00:14:53,NaN,NaN
4,2018-01-01 00:27:36,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,06/26/2002,A&B DOMESTIC NO 209A IN EFFECT,57 SPRINGFIELD ST,42.699340,-71.156938,...,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,Preventive Policing,2018,Non-Serious,06/26/2002,2018-01-01 00:27:36,2002-06-26,2018-01-01 00:27:36,NaN,a&b domestic no 209a in effect


Shape: (428527, 21)
Columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'latitude', 'longitude', 'geocode_confidence', 'person_id', 'category', 'Year', 'crime_severity', '_DOB_raw', '_Date_raw', 'DOB_dt', 'Date_dt', 'statutes', 'cleaned_charges']
Rows with charges: 4631


,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,latitude,longitude,...,person_id,category,Year,crime_severity,_DOB_raw,_Date_raw,DOB_dt,Date_dt,statutes,cleaned_charges
0,2018-01-01 00:01:14,NOISE ORD,3 HARRIMAN ST,Yes,NaN,07/03/1974,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,3 HARRIMAN ST,42.718522,-71.148148,...,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,Public Disturbances,2018,Non-Serious,07/03/1974,2018-01-01 00:01:14,1974-07-03,2018-01-01 00:01:14,NaN,a&b on family / household member / intimate pa...
1,2018-01-01 00:08:38,LOUD NOISE,1 HARRIMAN ST,Yes,NaN,01/21/1979,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,1 HARRIMAN ST FL 2,42.718522,-71.148148,...,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,Public Disturbances,2018,Non-Serious,01/21/1979,2018-01-01 00:08:38,1979-01-21,2018-01-01 00:08:38,NaN,a&b on family / household member / intimate pa...
2,2018-01-01 00:11:17,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,16 ALLEN ST,42.710782,-71.151911,...,NaN,Fire and Arson Incidents,2018,Non-Serious,NaN,2018-01-01 00:11:17,NaN,2018-01-01 00:11:17,NaN,NaN
3,2018-01-01 00:14:53,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,11 SUMMER ST,42.711117,-71.153015,...,NaN,Public Disturbances,2018,Non-Serious,NaN,2018-01-01 00:14:53,NaN,2018-01-01 00:14:53,NaN,NaN
4,2018-01-01 00:27:36,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,06/26/2002,A&B DOMESTIC NO 209A IN EFFECT,57 SPRINGFIELD ST,42.699340,-71.156938,...,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,Preventive Policing,2018,Non-Serious,06/26/2002,2018-01-01 00:27:36,2002-06-26,2018-01-01 00:27:36,NaN,a&b domestic no 209a in effect


### Build charge_class lookup

Load `unique_charges_standardized.csv` and build a dict: `base_charge → charge_class`.

In [19]:
#gffi
lookup_df = pd.read_csv(LOOKUP_PATH)

charge_class_lookup = dict(
    zip(
        lookup_df["base_charge"],
        lookup_df["charge_class"]
    )
)

print(f"Lookup entries: {len(charge_class_lookup)}")
print(f"Classes present: {sorted(lookup_df['charge_class'].dropna().unique())}")

print()
for sample in [
    "trespass",
    "murder",
    "firearm, carry without license",
    "drug, possess class a",
]:
    print(f"{sample!r} -> {charge_class_lookup.get(sample, 'NOT FOUND')}")

Lookup entries: 394
Classes present: ['Either', 'Felony', 'Misdemeanor']

'trespass' -> Misdemeanor
'murder' -> Felony
'firearm, carry without license' -> Either
'drug, possess class a' -> Misdemeanor


In [20]:
 
lookup_df = pd.read_csv(LOOKUP_PATH)
charge_class_lookup = dict(zip(lookup_df["base_charge"], lookup_df["charge_class"]))

print(f"Lookup entries: {len(charge_class_lookup)}")
print(f"Classes present: {set(charge_class_lookup.values())}")

# Quick sanity checks
print()
for sample in ["trespass", "murder", "firearm, carry without license", "drug, possess class a"]:
    print(f"  {sample!r} -> {charge_class_lookup.get(sample, 'NOT FOUND')}")

Lookup entries: 394
Classes present: {'Misdemeanor', 'Either', 'Felony'}

  'trespass' -> Misdemeanor
  'murder' -> Felony
  'firearm, carry without license' -> Either
  'drug, possess class a' -> Misdemeanor


### Compute `has_warrant`

Check the original `Charges` column. Split each row's charges by `;` and run `extract_warrant_type()` on each individual charge. If any charge had a warrant prefix, the row is flagged `True`.

In [21]:
## old 
def row_has_warrant(raw_charges):
    """Return True if any individual charge in the raw Charges string
    carries a warrant prefix. Returns NaN for rows with no charges."""
    if pd.isna(raw_charges):
        return np.nan
    parts = [p.strip() for p in str(raw_charges).split(";") if p.strip()]
    for part in parts:
        warrant_type, _ = extract_warrant_type(part)
        if warrant_type != "none":
            return True
    return False


df["has_warrant"] = df["cleaned_charges"].apply(row_has_warrant)

warrant_counts = df["has_warrant"].value_counts(dropna=False)
print("has_warrant value counts:")
print(warrant_counts)

# Spot-check
print("\nSample warrant rows:")
sample_warrant = df[df["has_warrant"] == True][["Charges", "cleaned_charges", "has_warrant"]].head(5)
for _, row in sample_warrant.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['cleaned_charges']}")
    print()

has_warrant value counts:
has_warrant
NaN      423896
False      2846
True       1785
Name: count, dtype: int64

Sample warrant rows:
  Charges: UNLICENSED OPERATION OF MV c90 S10; ATTACHING WRONG MV PLATES; RECEIVE STOLEN PROPERTY -$1200 c266 S
  Standardized: unlicensed operation of mv; attaching wrong mv plates; receive stolen property -$1200; warrant charges: standard warrant: registration sticker not; standard warrant: uninsured mv/trailer; standard warrant: equipment violation, miscellaneous mv; standard warrant: unregistered motor vehicle; standard warrant: unlicensed operation of mv

  Charges: STANDARD WARRANT: ASSAULT W/DANGEROUS; WEAPON c265 S15B; STANDARD WARRANT: THREAT TO COMMIT CRIME c2
  Standardized: standard warrant: assault w/dangerous; weapon; standard warrant: threat to commit crime; standard warrant: trespass

  Charges: STANDARD WARRANT: A&B WITH DANGEROUS WEAPON; c265 S15A; STANDARD WARRANT: LARCENY FROM PERSON c266 S
  Standardized: standard warrant: a&b with d

### Compute `is_misdemeanor`

Split `standardized_charges` by `"; "` and look up each charge's class in the lookup dict. A row is `True` only when **every** charge in the row classifies as `Misdemeanor`. Any `Felony`, `Either`, or `Unknown` charge makes the row `False`. Rows with no charges return `NaN`.

In [22]:
#gffi 
def row_is_misdemeanor(std_charges):
    """Return True if every standardized charge in the row is Misdemeanor.
    Returns False if any charge is Felony, Either, or Unknown.
    Returns NaN for rows with no charges."""
    if pd.isna(std_charges):
        return np.nan
    parts = [p.strip() for p in str(std_charges).split(";") if p.strip()]
    if not parts:
        return np.nan
    for charge in parts:
        cls = charge_class_lookup.get(charge, "Unknown")
        if cls != "Misdemeanor":
            return False
    return True


df["is_misdemeanor"] = df["cleaned_charges"].apply(row_is_misdemeanor)

misdemeanor_counts = df["is_misdemeanor"].value_counts(dropna=False)
print("is_misdemeanor value counts:")
print(misdemeanor_counts)

# Spot-check True rows
print("\nSample misdemeanor-only rows:")
sample_misd = df[df["is_misdemeanor"] == True][["Charges", "cleaned_charges", "is_misdemeanor"]].head(5)
for _, row in sample_misd.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['cleaned_charges']}")
    print()

# Spot-check False rows (not pure misdemeanor)
print("Sample non-misdemeanor rows:")
sample_non = df[df["is_misdemeanor"] == False][["Charges", "cleaned_charges", "is_misdemeanor"]].head(5)
for _, row in sample_non.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['cleaned_charges']}")
    print()

is_misdemeanor value counts:
is_misdemeanor
NaN      423896
False      3159
True       1472
Name: count, dtype: int64

Sample misdemeanor-only rows:
  Charges: A&B DOMESTIC NO 209A IN EFFECT
  Standardized: a&b domestic no 209a in effect

  Charges: USE MV WITHOUT AUTHORITY c90 S24; LARCENY UNDER $250 c266 S30
  Standardized: use mv without authority; larceny under $250

  Charges: DRUG, POSSESS CLASS B c94C S34
  Standardized: drug, possess class b

  Charges: LICENSE SUSPENDED, OP MV WITH c90 S23
  Standardized: license suspended, op mv with

  Charges: DRUG, POSSESS CLASS B c94C S34
  Standardized: drug, possess class b

Sample non-misdemeanor rows:
  Charges: A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PARTNE; STRANGULATION OR SUFFOCATION
  Standardized: a&b on family / household member / intimate partne; strangulation or suffocation

  Charges: A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PARTNE
  Standardized: a&b on family / household member / intimate partne

  Charges: WITNESS,

### Diagnose any charges not found in lookup

Identifies standardized charges that appear in the data but are missing from `unique_charges_standardized.csv`.

In [23]:
all_std = (
    df["cleaned_charges"]
    .dropna()
    .str.split("; ")
    .explode()
    .str.strip()
    .unique()
)

missing = [c for c in all_std if c and c not in charge_class_lookup]
if missing:
    print(f"{len(missing)} standardized charges not found in lookup:")
    for c in sorted(missing):
        print(f"  {c!r}")
else:
    print("All cleaned charges found in lookup.")

995 standardized charges not found in lookup:
  '$37c(c)'
  '& $30(1)'
  '(1)'
  '500 ft of bldg'
  'a&b in the presence of a po'
  'a&b on family / household member / intimate partne'
  'a&b on police officer c265 s$13d'
  'a&b with dangerous weapon 265 si5a'
  'a&b with dangerous weapon c265 si5a'
  'address: 126 franklin st'
  'address: 155 salem st'
  'address: 20 portland st'
  'address: 205 broadway'
  'address: 25 foster st'
  'address: 3 s bowdoin st'
  'address: 30 manchester st'
  'address: 300 howard st'
  'address: 32 manchester st'
  'address: 590 broadway'
  'address: 62 bowdoin st'
  'address: 63 newbury st'
  'address: 75 haverhill st'
  'alcohol from open container in mv, drink ]'
  'assault w/dangerous weapon 265 si5b'
  'assault w/dangerous weapon c265 si5b'
  'bench warrant: a&b on family / household member / intimate'
  'bench warrant: attempt to commit crime'
  'bench warrant: b&e daytime for felony, person in fear'
  'bench warrant: contempt, criminal (common law

### Verify & summarize

In [24]:
charged_rows = df[df["cleaned_charges"].notna()]

print(f"Total rows: {len(df):,}")
print(f"Rows with charges: {len(charged_rows):,}")
print()
print(f"has_warrant = True  : {(df['has_warrant'] == True).sum():,}")
print(f"has_warrant = False : {(df['has_warrant'] == False).sum():,}")
print(f"has_warrant = NaN   : {df['has_warrant'].isna().sum():,}")
print()
print(f"is_misdemeanor = True  : {(df['is_misdemeanor'] == True).sum():,}")
print(f"is_misdemeanor = False : {(df['is_misdemeanor'] == False).sum():,}")
print(f"is_misdemeanor = NaN   : {df['is_misdemeanor'].isna().sum():,}")
print()
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Total rows: 428,527
Rows with charges: 4,631

has_warrant = True  : 1,785
has_warrant = False : 2,846
has_warrant = NaN   : 423,896

is_misdemeanor = True  : 1,472
is_misdemeanor = False : 3,159
is_misdemeanor = NaN   : 423,896

Final shape: (428527, 23)
Columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'latitude', 'longitude', 'geocode_confidence', 'person_id', 'category', 'Year', 'crime_severity', '_DOB_raw', '_Date_raw', 'DOB_dt', 'Date_dt', 'statutes', 'cleaned_charges', 'has_warrant', 'is_misdemeanor']


### Save checkpoint15

In [26]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved checkpoint15: {OUT_PATH}")
print(f"Shape: {df.shape}")

Saved checkpoint15: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/charges/../data/missing_dates_csv/md_checkpoints/checkpoint15_misdemeanor_warrant.csv
Shape: (428527, 23)
